# Numeric Stability and Speed of B-Splines
We build tables where we report (in terms of the degree of B-splines) the accuracy and timing of the exact computation of B-spline values against their computations made by three other approaches: the classic approach; the De Boor's approach; and the approach proposed in the ``splinekit`` library.

We examine three regimes. In the *critical regime*, we place our focus on those places where the accuracy degrades the most when the degree rises. In the *global regime*, we do not focus on specific places. At first we explore low spline degrees, which allows us to experiment with the De Boor's approach; then, in the interest of time, we discard the De Boor's approach, which allows us to experiment with higher degrees.

## Caveat
The results provide representative timings only when the notebook is run natively with a properly installed ``splinekit`` library.

## Critical Regime
At first, the accuracy is measured at random samples taken over the left and right ends of the tails of the B-splines; those are the locations where the degradation of the computations due to rounding and truncation errors is most pronounced.

In [ ]:
# Load the required libraries
from fractions import Fraction
import math
import numpy as np
import time

import splinekit as sk # This library

# Experimental conditions
highest_degree = 16 # Computation time shows a steep increase with the degree
size = 2 * 200 # Number of random arguments
tail_chunk = 2 # Outward support where the computations are performed

# Initialize the generator of random numbers
rng = np.random.default_rng()

# Ground-truth arguments
# The most critical part is near the ends of the tails of the B-spline
random_rational_args = [
    np.concatenate((
        # Support of length 'tail_chunk' near the left end of the tail
        [
            Fraction(x).limit_denominator(2 ** 48)
            for x in rng.uniform(
                low = -0.5 * (degree + 1),
                high = -0.5 * (degree + 1) + tail_chunk,
                size = size // 2
            )
        ],
        # Support of length 'tail_chunk' near the right end of the tail
        [
            Fraction(x).limit_denominator(2 ** 48)
            for x in rng.uniform(
                low = 0.5 * (degree + 1) - tail_chunk,
                high = 0.5 * (degree + 1),
                size = size // 2
            )
        ]
    ))
    for degree in range(highest_degree + 1)
]

# Populate the cache of splinekit
# The cache is persistent throughout the life of the kernel
# This step is done here only to favor a consistency in timings; it is functionally neutral
for degree in range(highest_degree + 1):
    sk.b_spline(0, degree)

# First approach: ground truth
ground_truth = []
for (n, xn) in enumerate(random_rational_args):
    start = time.perf_counter()
    data = [sk.spline_utilities._db_frac(x, n, 0) for x in xn]
    end = time.perf_counter()
    ground_truth += [(data, end - start)]

# Second approach: compute B-splines as in (2) with float accuracy
classic = []
for (n, xn) in enumerate(random_rational_args):
    start = time.perf_counter()
    data = [sk.spline_utilities._db(float(x), n, 0) for x in xn]
    end = time.perf_counter()
    classic += [(data, end - start)]

# De Boor's relation
def de_boor_b_spline (
    x: float,
    n: int
) -> float:
    if 0 == n:
        # Initial condition of the recurrence relation
        return (sk.polynomial_simple_element(x + 0.5, 0) -
            sk.polynomial_simple_element(x - 0.5, 0))
    # Recurrence relation
    return (((x + 0.5 * (n + 1)) * de_boor_b_spline(x + 0.5, n - 1) -
        (x - 0.5 * (n + 1)) * de_boor_b_spline(x - 0.5, n - 1)) / n)

# Third approach: De Boor's recurrence with float accuracy
de_boor = []
for (n, xn) in enumerate(random_rational_args):
    start = time.perf_counter()
    data = [de_boor_b_spline(float(x), n) for x in xn]
    end = time.perf_counter()
    de_boor += [(data, end - start)]

# Fourth approach: splinekit library, with float accuracy
sk_library = []
for (n, xn) in enumerate(random_rational_args):
    start = time.perf_counter()
    data = [sk.b_spline(float(x), n) for x in xn]
    end = time.perf_counter()
    sk_library += [(data, end - start)]

# Mean-square error
def mse (
    gt: [Fraction],
    data: [float]
) -> float:
    error = [
        (float(frac) - data[k]) ** 2
        for (k, frac) in enumerate(gt)
    ]
    return math.fsum(error)

# Signal-to-noise ratio in dB units
def snr_db (
    gt: [Fraction],
    data: [float]
) -> float:
    signal = mse(gt, np.zeros(size, dtype = float))
    noise = mse(gt, data)
    return 10.0 * math.log10(signal / noise) if 0.0 != noise else float("inf")

# Table of results
print()
print("Critical Regime, Tail of B-Splines")
print()
print("==================================================================================")
print("       |   Ground-Truth   |     Classic      |     De Boor      |    Splinekit    ")
print("Degree | SNR[dB]  Time[s] | SNR[dB]  Time[s] | SNR[dB]  Time[s] | SNR[dB]  Time[s]")
print("-------+------------------+------------------+------------------+-----------------")
for (n, gtn) in enumerate(ground_truth):
    print("    {0:2d} | {1:7.1f}  {2:7.1e} | {3:7.1f}  {4:7.1e} | {5:7.1f}  {6:7.1e} | {7:7.1f}  {8:7.1e}".format(
        n,
        snr_db(gtn[0], ground_truth[n][0]),
        ground_truth[n][1],
        snr_db(gtn[0], classic[n][0]),
        classic[n][1],
        snr_db(gtn[0], de_boor[n][0]),
        de_boor[n][1],
        snr_db(gtn[0], sk_library[n][0]),
        sk_library[n][1]
    ))
print("==================================================================================")


###################
#                 #
#   Be patient!   #
#                 #
###################

# Duration of the native computations on a desktop computer of year 2021: ~20s
# Duration of the WebAssembly computations: about twice




## Global Regime, Low Degrees
In the global regime, the accuracy is measured at random samples taken over the whole support of the B-splines. In the next table, we include the slow De Boor's approach but limit the table to low degrees.

In [ ]:
# Load the required libraries
from fractions import Fraction
import math
import numpy as np
import time

import splinekit as sk # This library

# Experimental conditions
highest_degree = 16 # Computation time shows a steep increase with the degree
size = 400 # Number of random arguments

# Initialize the generator of random numbers
rng = np.random.default_rng()

# Ground-truth arguments
random_rational_args = [
    [
        Fraction(x).limit_denominator(2 ** 48)
        for x in rng.uniform(
            low = -0.5 * (degree + 1),
            high = 0.5 * (degree + 1),
            size = size
        )
    ]
    for degree in range(highest_degree + 1)
]

# Populate the cache of splinekit
# The cache is persistent throughout the life of the kernel
# This step is done here only to favor a consistency in timings; it is functionally neutral
for degree in range(highest_degree + 1):
    sk.b_spline(0, degree)

# First approach: ground truth
ground_truth = []
for (n, xn) in enumerate(random_rational_args):
    start = time.perf_counter()
    data = [sk.spline_utilities._db_frac(x, n, 0) for x in xn]
    end = time.perf_counter()
    ground_truth += [(data, end - start)]

# Second approach: compute B-splines as in (2) with float accuracy
classic = []
for (n, xn) in enumerate(random_rational_args):
    start = time.perf_counter()
    data = [sk.spline_utilities._db(float(x), n, 0) for x in xn]
    end = time.perf_counter()
    classic += [(data, end - start)]

# De Boor's relation
def de_boor_b_spline (
    x: float,
    n: int
) -> float:
    if 0 == n:
        # Initial condition of the recurrence relation
        return (sk.polynomial_simple_element(x + 0.5, 0) -
            sk.polynomial_simple_element(x - 0.5, 0))
    # Recurrence relation
    return (((x + 0.5 * (n + 1)) * de_boor_b_spline(x + 0.5, n - 1) -
        (x - 0.5 * (n + 1)) * de_boor_b_spline(x - 0.5, n - 1)) / n)

# Third approach: De Boor's recurrence with float accuracy
de_boor = []
for (n, xn) in enumerate(random_rational_args):
    start = time.perf_counter()
    data = [de_boor_b_spline(float(x), n) for x in xn]
    end = time.perf_counter()
    de_boor += [(data, end - start)]

# Fourth approach: splinekit library, with float accuracy
sk_library = []
for (n, xn) in enumerate(random_rational_args):
    start = time.perf_counter()
    data = [sk.b_spline(float(x), n) for x in xn]
    end = time.perf_counter()
    sk_library += [(data, end - start)]

# Mean-square error
def mse (
    gt: [Fraction],
    data: [float]
) -> float:
    error = [
        (float(frac) - data[k]) ** 2
        for (k, frac) in enumerate(gt)
    ]
    return math.fsum(error)

# Signal-to-noise ratio in dB units
def snr_db (
    gt: [Fraction],
    data: [float]
) -> float:
    signal = mse(gt, np.zeros(size, dtype = float))
    noise = mse(gt, data)
    return 10.0 * math.log10(signal / noise) if 0.0 != noise else float("inf")

# Table of results
print()
print("Global Regime, Low Degrees")
print()
print("==================================================================================")
print("       |   Ground-Truth   |     Classic      |     De Boor      |    Splinekit    ")
print("Degree | SNR[dB]  Time[s] | SNR[dB]  Time[s] | SNR[dB]  Time[s] | SNR[dB]  Time[s]")
print("-------+------------------+------------------+------------------+-----------------")
for (n, gtn) in enumerate(ground_truth):
    print("    {0:2d} | {1:7.1f}  {2:7.1e} | {3:7.1f}  {4:7.1e} | {5:7.1f}  {6:7.1e} | {7:7.1f}  {8:7.1e}".format(
        n,
        snr_db(gtn[0], ground_truth[n][0]),
        ground_truth[n][1],
        snr_db(gtn[0], classic[n][0]),
        classic[n][1],
        snr_db(gtn[0], de_boor[n][0]),
        de_boor[n][1],
        snr_db(gtn[0], sk_library[n][0]),
        sk_library[n][1]
    ))
print("==================================================================================")


###################
#                 #
#   Be patient!   #
#                 #
###################

# Duration of the native computations on a desktop computer of year 2021: ~20s
# Duration of the WebAssembly computations: about twice




## Global Regime
Finally, the accuracy is again measured at random samples taken over the whole support of the B-splines. In the next table, we discard the De Boor's approach but raise degrees until the classic approach reaches a point where there is about as much noise as signal.

In [ ]:
# Load the required libraries
from fractions import Fraction
import math
import numpy as np
import time

import splinekit as sk # This library

# Experimental conditions
highest_degree = 31 # Computation time shows a steep increase with the degree
size = 400 # Number of random arguments

# Initialize the generator of random numbers
rng = np.random.default_rng()

# Ground-truth arguments
random_rational_args = [
    [
        Fraction(x).limit_denominator(2 ** 48)
        for x in rng.uniform(
            low = -0.5 * (degree + 1),
            high = 0.5 * (degree + 1),
            size = size
        )
    ]
    for degree in range(highest_degree + 1)
]

# Populate the cache of splinekit
# The cache is persistent throughout the life of the kernel
# This step is done here only to favor a consistency in timings; it is functionally neutral
for degree in range(highest_degree + 1):
    sk.b_spline(0, degree)

# First approach: ground truth
ground_truth = []
for (n, xn) in enumerate(random_rational_args):
    start = time.perf_counter()
    data = [sk.spline_utilities._db_frac(x, n, 0) for x in xn]
    end = time.perf_counter()
    ground_truth += [(data, end - start)]

# Second approach: compute B-splines as in (2) with float accuracy
classic = []
for (n, xn) in enumerate(random_rational_args):
    start = time.perf_counter()
    data = [sk.spline_utilities._db(float(x), n, 0) for x in xn]
    end = time.perf_counter()
    classic += [(data, end - start)]

# Fourth approach: splinekit library, with float accuracy
sk_library = []
for (n, xn) in enumerate(random_rational_args):
    start = time.perf_counter()
    data = [sk.b_spline(float(x), n) for x in xn]
    end = time.perf_counter()
    sk_library += [(data, end - start)]

# Mean-square error
def mse (
    gt: [Fraction],
    data: [float]
) -> float:
    error = [
        (float(frac) - data[k]) ** 2
        for (k, frac) in enumerate(gt)
    ]
    return math.fsum(error)

# Signal-to-noise ratio in dB units
def snr_db (
    gt: [Fraction],
    data: [float]
) -> float:
    signal = mse(gt, np.zeros(size, dtype = float))
    noise = mse(gt, data)
    return 10.0 * math.log10(signal / noise) if 0.0 != noise else float("inf")

# Table of results
print()
print("Global Regime")
print()
print("===============================================================")
print("       |   Ground-Truth   |     Classic      |    Splinekit    ")
print("Degree | SNR[dB]  Time[s] | SNR[dB]  Time[s] | SNR[dB]  Time[s]")
print("-------+------------------+------------------+-----------------")
for (n, gtn) in enumerate(ground_truth):
    print("    {0:2d} | {1:7.1f}  {2:7.1e} | {3:7.1f}  {4:7.1e} | {5:7.1f}  {6:7.1e}".format(
        n,
        snr_db(gtn[0], ground_truth[n][0]),
        ground_truth[n][1],
        snr_db(gtn[0], classic[n][0]),
        classic[n][1],
        snr_db(gtn[0], sk_library[n][0]),
        sk_library[n][1]
    ))
print("===============================================================")


###################
#                 #
#   Be patient!   #
#                 #
###################

# Duration of the native computations on a desktop computer of year 2021: ~5s
# Duration of the WebAssembly computations: about twice


